# 语义分割算法-PSPNet

同济子豪兄 2023-2-13 6-11 6-25 7-12

## 进入MMSegmentation主目录

In [1]:
# import os
# os.chdir('mmsegmentation')

import os
# 假设 mmsegmentation 的绝对路径是 /project/mmsegmentation
mmsegmentation_path = "E:/bishe_custom_data/mmsegmentation-main"
# 切换到 mmsegmentation 文件夹
os.chdir(mmsegmentation_path)
# 验证当前工作目录
print("当前工作目录:", os.getcwd())  # 输出: /project/mmsegmentation

当前工作目录: E:\bishe_custom_data\mmsegmentation-main


## 载入config配置文件

PSPNet模型库：https://github.com/open-mmlab/mmsegmentation/tree/main/configs/pspnet

In [2]:
# from mmengine import Config
# 
# cfg = Config.fromfile('./configs/pspnet/pspnet_r50-d8_4xb2-40k_cityscapes-512x1024.py')
# dataset_cfg = Config.fromfile('./configs/_base_/datasets/ZihaoDataset_pipeline.py')
# cfg.merge_from_dict(dataset_cfg)

#训练公共数据集 cityspace
from mmengine import Config
# cfg = Config.fromfile('./configs/pspnet/pspnet_r18-d8_4xb2-80k_cityscapes-512x1024.py')
#换大一点的模型试试。
# cfg = Config.fromfile('./configs/pspnet/pspnet_r50b-d8_4xb2-80k_cityscapes-512x1024.py')
#换一个更深的
cfg = Config.fromfile('./configs/pspnet/pspnet_r101-d8_4xb2-amp-80k_cityscapes-512x1024.py')

dataset_cfg = Config.fromfile('./configs/_base_/datasets/ZihaoDataset_pipeline.py')
cfg.merge_from_dict(dataset_cfg)

## 修改config配置文件

In [3]:
# 类别个数
#NUM_CLASS = 6

# 类别个数
NUM_CLASS = 5

In [4]:
# cfg.model.data_preprocessor.size = cfg.crop_size
# 
# # 单卡训练时，需要把 SyncBN 改成 BN
# cfg.norm_cfg = dict(type='BN', requires_grad=True)
# cfg.model.backbone.norm_cfg = cfg.norm_cfg
# cfg.model.decode_head.norm_cfg = cfg.norm_cfg
# cfg.model.auxiliary_head.norm_cfg = cfg.norm_cfg
# 
# # 模型 decode/auxiliary 输出头，指定为类别个数
# cfg.model.decode_head.num_classes = NUM_CLASS
# cfg.model.auxiliary_head.num_classes = NUM_CLASS
# 
# # 训练 Batch Size
# cfg.train_dataloader.batch_size = 4
# 
# # 结果保存目录
# cfg.work_dir = './work_dirs/ZihaoDataset-PSPNet'
# 
# # 模型保存与日志记录
# cfg.train_cfg.max_iters = 40000 # 训练迭代次数
# cfg.train_cfg.val_interval = 500 # 评估模型间隔
# cfg.default_hooks.logger.interval = 100 # 日志记录间隔
# cfg.default_hooks.checkpoint.interval = 2500 # 模型权重保存间隔
# cfg.default_hooks.checkpoint.max_keep_ckpts = 1 # 最多保留几个模型权重
# cfg.default_hooks.checkpoint.save_best = 'mIoU' # 保留指标最高的模型权重
# 
# # 随机数种子
# cfg['randomness'] = dict(seed=0)



cfg.model.data_preprocessor.size = cfg.crop_size

# 单卡训练时，需要把 SyncBN 改成 BN
cfg.norm_cfg = dict(type='BN', requires_grad=True)
cfg.model.backbone.norm_cfg = cfg.norm_cfg
cfg.model.decode_head.norm_cfg = cfg.norm_cfg
cfg.model.auxiliary_head.norm_cfg = cfg.norm_cfg

# 模型 decode/auxiliary 输出头，指定为类别个数
cfg.model.decode_head.num_classes = NUM_CLASS
cfg.model.auxiliary_head.num_classes = NUM_CLASS

# 训练 Batch Size
cfg.train_dataloader.batch_size = 4

# 结果保存目录
cfg.work_dir = './work_dirs/ZihaoDataset-PSPNet'

# 模型保存与日志记录
cfg.train_cfg.max_iters = 5000 # 训练迭代次数
cfg.train_cfg.val_interval = 100 # 评估模型间隔
cfg.default_hooks.logger.interval = 10 # 日志记录间隔
cfg.default_hooks.checkpoint.interval = 10 # 模型权重保存间隔
cfg.default_hooks.checkpoint.max_keep_ckpts = 1 # 最多保留几个模型权重
cfg.default_hooks.checkpoint.save_best = 'mIoU' # 保留指标最高的模型权重

# 随机数种子
cfg['randomness'] = dict(seed=0)

# # # 加载本地预训练模型
# cfg.load_from = 'checkpoint/pspnet_r18-d8_512x1024_80k_cityscapes_20201225_021458-09ffa746.pth'  # 替换为你的预训练模型路径

## 查看完整config配置文件

In [5]:
print(cfg.pretty_text)

crop_size = (
    512,
    512,
)
data_preprocessor = dict(
    bgr_to_rgb=True,
    mean=[
        123.675,
        116.28,
        103.53,
    ],
    pad_val=0,
    seg_pad_val=255,
    size=(
        512,
        1024,
    ),
    std=[
        58.395,
        57.12,
        57.375,
    ],
    type='SegDataPreProcessor')
data_root = 'data/mmseg_custom_dataset/'
dataset_type = 'ZihaoDataset'
default_hooks = dict(
    checkpoint=dict(
        by_epoch=False,
        interval=10,
        max_keep_ckpts=1,
        save_best='mIoU',
        type='CheckpointHook'),
    logger=dict(interval=10, log_metric_by_epoch=False, type='LoggerHook'),
    param_scheduler=dict(type='ParamSchedulerHook'),
    sampler_seed=dict(type='DistSamplerSeedHook'),
    timer=dict(type='IterTimerHook'),
    visualization=dict(type='SegVisualizationHook'))
default_scope = 'mmseg'
env_cfg = dict(
    cudnn_benchmark=True,
    dist_cfg=dict(backend='nccl'),
    mp_cfg=dict(mp_start_method='fork', opencv_num_threads=0

## 保存最终的config配置文件

In [5]:
cfg.dump('Zihao-Configs/ZihaoDataset_PSPNet_20230818.py')